# 0923 18일차

## 1. learning_rate 지정

`optimizer`를 문자열로 주면 기본값이 쓰이므로, 직접 정하려면 객체로 넘겨야 함

### 1-1. 문자열과 객체

```python
model.compile(loss='mse', optimizer='adam')                         # 기본값 0.001
model.compile(loss='mse', optimizer=Adam(learning_rate=0.001))      # 직접 지정
```

1. 문자열 : 짧지만 기본값 외에는 못 씀
2. 객체 : `learning_rate` 외에 다른 인자도 넣을 수 있음

- Adam의 기본 `learning_rate`는 `0.001`

#### 주의) lr이 너무 클 때

digits에서 `learning_rate=0.02`(기본값의 20배)를 줬더니 학습이 되지 않음

```
loss  2.306183   ≈  ln(10) = 2.302585
acc   0.1        =  1/10
```

1. 클래스 10개에 확률을 1/10씩 균등하게 내놓는 상태
2. 보폭이 커서 최저점을 지나치고, 다음 갱신에서 더 멀어짐
3. 가중치가 발산해 출력이 전부 비슷해지고 softmax가 균등하게 나눠 줌

- 다중분류에서 loss가 `ln(클래스 수)` 근처면 찍는 것과 같다는 뜻. 클래스 10개는 2.30, 100개는 4.61
- `batch_size=1`과 겹치면 더 심함. 사진 한 장마다 크게 움직이므로 방향이 매번 뒤집힘
- 기본값부터 시작해 한 칸씩 옮기며 비교하는 것이 순서

## 2. ReduceLROnPlateau

`val_loss`가 나아지지 않으면 `learning_rate`를 자동으로 줄여 주는 콜백

**필요한 이유**

1. `lr`이 고정이면 바닥 근처에서 보폭이 충분히 줄지 않아 진동함
2. 큰 `lr`로 빠르게 내려온 뒤 바닥에서 줄이면 둘 다 얻을 수 있음

```python
rlr = ReduceLROnPlateau(
    monitor='val_loss',
    mode='auto',
    patience=20,
    factor=0.5,
    verbose=1,
)

model.fit(..., callbacks=[es, rlr])
```

### 2-1. 인자

1. `monitor` : 무엇을 보고 판단할지. 보통 `val_loss`
2. `mode` : `min`이면 작아져야 개선. `auto`는 `monitor` 이름으로 자동 판단
3. `patience` : 개선이 없는 epoch를 몇 번까지 참을지
4. `factor` : 줄일 비율. `0.5`면 절반으로
5. `verbose` : `1`이면 줄일 때마다 메시지를 출력

- `factor`를 너무 작게 주면 `lr`이 금방 0에 가까워져 학습이 멈춤
- 만들어만 두고 `callbacks`에 안 넘기면 동작하지 않음

### 2-2. EarlyStopping과 patience

두 콜백이 같은 `val_loss`를 보므로 **`patience`에 차이를 둬야 함**

1. `ReduceLROnPlateau`의 `patience`를 짧게
2. `EarlyStopping`의 `patience`를 길게

- 같거나 `EarlyStopping`이 더 짧으면 `lr`을 줄여 보기도 전에 훈련이 끝남
- 줄인 `lr`로 다시 내려갈 기회를 주려면 그만큼 더 참아야 함

## 3. RNN

Recurrent Neural Network. 순서가 있는 데이터를 **한 칸씩 순서대로** 넣으면서, 앞 칸의 결과를 다음 칸 계산에 같이 넣는 신경망

### 3-1. 순환

![접힌 모습과 시간 순서로 펼친 모습, 가운데 칸을 열어 units 세 개가 입력 하나와 앞 결과 세 개를 모두 받는 모습, 각 입력이 출력까지 지나는 칸 수만큼 순환 가중치가 곱해져 옛날 입력의 영향이 줄거나 커지는 모습](assets/rnn-recurrent.svg)

1. 한 번에 전부 넣지 않고 순서대로 하나씩 넣음
2. 앞 칸에서 나온 결과를 다음 칸에 함께 넣음
3. h는 숫자 하나가 아니라 units 개수만큼의 벡터

- `밥을`을 계산할 때 `나는`을 기억하고 있으므로 순서가 의미를 갖는 데이터를 다룰 수 있음
- 같은 가중치를 칸마다 다시 쓰기 때문에 `Recurrent`(순환)라고 부름
- **units마다 다른 입력을 받는 것이 아님.** 같은 입력에 서로 다른 가중치를 곱함

### 3-2. timesteps

시퀀스를 몇 칸으로 끊을지. 문장이 단어 3개면 `timesteps`는 3

```
"나는 밥을 먹었다"    timestep 1 : 나는
                     timestep 2 : 밥을
                     timestep 3 : 먹었다
```

- 몇 칸까지 거슬러 볼지를 정하는 값이라, 길면 더 멀리 참조하지만 계산이 늘어남

### 3-3. 입력 차원

| 모델 | 입력 | 차원 |
|---|---|---|
| DNN | `(장수, feature)` | 2차원 |
| RNN | `(장수, timesteps, feature)` | 3차원 |
| CNN | `(장수, 세로, 가로, 채널)` | 4차원 |

```python
model.add(SimpleRNN(units=10, input_shape=(3, 1)))
model.add(SimpleRNN(units=10, input_length=3, input_dim=1))   # 같은 뜻
```

1. `input_shape=(timesteps, feature)`로 한 번에 주거나
2. `input_length`와 `input_dim`으로 나눠 줘도 됨

- `(100, 3, 1)`이면 문장 100개, 각 3칸, 한 칸을 숫자 1개로 표현한 것
- 가운데 자리가 순서를 담당하는 축
- RNN은 3차원을 받아 `(장수, units)` **2차원으로 내보냄.** `Dense`에 바로 연결됨
- CNN은 4차원을 내보내 `Flatten`이나 `GAP`이 필요했지만 RNN은 필요 없음. timesteps 축이 사라지기 때문

### 3-4. 가중치

가중치가 두 벌임. **입력 가중치**는 지금 입력에, **순환 가중치**는 앞 칸 결과에 곱해짐

![units 하나에 입력이 feature 개수만큼, 앞 결과가 units 개수만큼, 바이어스 하나가 들어오는 그림. 그런 units이 units 개수만큼 있으므로 파라미터는 units 곱하기 괄호 feature 더하기 units 더하기 1. 가중치 모양과 한 칸의 계산식을 함께 표시](assets/rnn-weights.svg)

1. 첫 칸의 앞 결과는 0에서 시작함. 앞이 없기 때문
2. **timesteps 전체가 가중치 한 벌을 공유함.** 칸마다 새로 만들지 않고 같은 것을 다시 씀
3. `return_sequences=False`가 기본이라 마지막 칸의 결과만 내보냄

- 마지막 결과에 앞 칸들의 내용이 누적돼 있으므로 하나만 내보내도 됨
- 출력이 `(장수, units)`인 이유. timesteps 축이 사라짐

#### 파라미터 개수

units **하나**를 계산할 때 들어오는 선을 세면 `feature + units + 1`개. 그런 units이 units 개수만큼 있음

- 순환 가중치가 가장 큰 이유는 units끼리의 연결이라 양쪽이 다 units 개수이기 때문
- Dense는 `(입력 + 1) × 노드수`. RNN은 입력 자리에 `feature + units`이 들어감. **앞 칸 결과도 입력처럼 들어오는 것**
- **timesteps는 파라미터에 들어가지 않음.** `(3, 1)`이든 `(30, 1)`이든 `SimpleRNN(10)`은 120개로 같음. 늘어나는 것은 계산 횟수뿐

#### 펼쳐 보기

![세로 화살표 세 개는 모두 같은 입력 가중치이고 가로 화살표 두 개는 모두 같은 순환 가중치다. 마지막 입력은 입력 가중치만 거치고 그 앞 입력은 순환 가중치를 한 번, 첫 입력은 두 번 더 거친다](assets/rnn-unfold.svg)

1. 세로 화살표는 전부 같은 입력 가중치, 가로 화살표는 전부 같은 순환 가중치
2. 출력까지 가는 동안 **거치는 순환 가중치 수만큼** 곱해짐
3. 옛날 입력일수록 멀리서 출발하므로 더 여러 번 거침

- 1보다 작으면 앞 내용이 거의 사라지고, 1보다 크면 커짐
- timesteps가 30이면 첫 입력은 순환 가중치를 29번 거침
- `SimpleRNN`이 긴 시퀀스에 약한 이유. 역방향으로 미분할 때도 같은 곱이 쌓임

### 3-5. LSTM

Long Short-Term Memory. `SimpleRNN`이 **긴 시퀀스에서 앞부분을 잊는 문제**를 고치려고 나온 층

**필요한 이유**

1. `SimpleRNN`은 `h` 하나로 모든 것을 기억하려다 보니 순환 가중치의 곱이 쌓여 앞부분이 사라짐
2. 기억을 나르는 통로를 따로 두면 곱이 쌓이는 경로를 피할 수 있음

```
SimpleRNN   h 하나만 다음 칸으로
LSTM        h + cell state 두 개가 다음 칸으로
```

- `cell state`가 기억 전용 통로. 곱이 아니라 더하기로 흘러가서 여러 칸을 지나도 잘 사라지지 않음
- 잊는 것과 넣는 것을 게이트로 나눠 정함. forget(무엇을 버릴지), input(무엇을 넣을지), output(무엇을 내보낼지)

#### 파라미터 개수 비교

게이트 3개와 새로 넣을 후보값 1개, 합쳐서 **가중치가 4벌** 필요하므로 `SimpleRNN`의 4배

| 층 | 파라미터 | 배수 |
|---|---|---|
| `SimpleRNN` | 120 | 1배 |
| `LSTM` | 480 | 4배 |
| `GRU` | 390 | 3배 |

`units=10`, `input_shape=(3, 1)` 기준

```
4 × units × (feature + units + 1)  =  4 × 10 × (1 + 10 + 1)  =  480
```

- `GRU`는 게이트를 2개로 줄인 것이라 3배. 성능은 비슷하면서 가벼움
- 파라미터가 늘어난 만큼 학습이 느려지므로, 짧은 시퀀스에는 `SimpleRNN`으로 충분한 경우도 있음

## 4. 시계열 데이터

값 하나짜리 수열만 있고 **정답 컬럼이 없음**. 앞 몇 개로 다음 하나를 맞히도록 사람이 잘라서 만듦

- 정형 데이터는 csv에 특성 컬럼과 정답 컬럼이 나뉘어 있지만, 주가·기온 같은 수열은 그렇지 않음
- 내일 주가라는 정답은 아직 존재하지 않으므로, 과거 안에서 정답 자리를 정해야 함

### 4-1. y 만들기

한 칸씩 밀면서 앞 `timesteps`개를 x, 그다음 하나를 y로 떼어냄

```
원본  [1 2 3 4 5 6 7 8 9 10]     timesteps=3

     x          y
  [1 2 3]  →   4
  [2 3 4]  →   5
  [3 4 5]  →   6
     ...
```

1. 같은 값이 여러 행에 겹쳐 들어감. `3`은 첫 행에서 마지막 자리, 둘째 행에서 가운데로 쓰임
2. 마지막 `timesteps`개는 뒤에 정답이 없어 버려짐

- 행 개수는 `전체 길이 - timesteps`. 위 예는 `10 - 3 = 7`행
- RNN에 넣으려면 `(장수, timesteps, 특성)` 3차원으로 맞춤. 값이 하나뿐이면 마지막에 `1`을 붙이고, 시가·종가·거래량을 같이 쓰면 `3`

### 4-2. timesteps 선택

며칠치를 보고 맞힐지. 이 값에 따라 문제 자체가 달라짐

1. 작으면 : 행이 많지만 멀리 못 봄
2. 크면 : 멀리 보지만 행이 줄고 계산이 늘어남

- 정답이 주어진 데이터가 아니므로 정해진 값이 없음. 바꿔 가며 비교해야 함
- 데이터가 짧을수록 크게 잡기 어려움. 길이 100에 `timesteps=30`이면 70행만 남음

### 4-3. 예측할 입력 만들기

훈련에 쓴 x와 **차원을 맞춰야 함**

```python
x_predict = np.array([8, 9, 10]).reshape(1, 3, 1)
y_predict = model.predict(x_predict)
```

1. 훈련 x가 `(7, 3, 1)`이므로 한 건만 넣어도 `(1, 3, 1)`
2. 맨 앞이 장수 자리. 한 건이면 1

- 자를 때 정한 timesteps와 같아야 함. 3개씩 잘랐으면 예측도 3개를 줌
- 수열의 마지막 구간을 넣으면 그다음 값을 예측하는 셈